# LSTM Phenology Model — Yedoensis Cherry Blossom

Trains an LSTM on the GMU Somei-yoshino (*Cerasus yedoensis*) cherry blossom dataset,
evaluates its performance, and then explores how its learned behaviour compares to
process-based models when applied to synthetic temperature series across a range of
winter temperatures — replicating the sensitivity plot from `model_exploration.ipynb`
with the LSTM added as an additional curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import torch
import torch.nn.functional as F
from typing import Callable, Dict, List, Optional, Tuple

from pysephone.constants import KEY_FEATURES, KEY_OBSERVATIONS_INDEX
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.evaluation.regression import SingleTargetRegression
from pysephone.models.lstm import LSTMModel
from pysephone.models.random_forest import RandomForestModel
from pysephone.models.unimodal_hybrid import UnimodalHybridModel
from pysephone.models.util.func_phenology import (
    func_utah_chill,
    func_chilling_days,
    func_dynamic_chill_daily,
)

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
OBS_KEY      = 'gmu_0'
CUTOFF       = 2015          # train: 1986–2014, test: 2015–2021
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# Shared style
PALETTE = {
    'temp_warm': '#f4a46280',
    'temp_cold': '#7eb8d480',
    'temp_line': '#c0392b',
    'zero':      '#cccccc',
    'thresh':    '#555555',
}
MODEL_COLORS = {
    'GDD':              '#888888',
    'Utah+GDD':         '#2980b9',
    'ChillingDays+GDD': '#27ae60',
    'Dynamic+GDD':      '#8e44ad',
    'LSTM':             '#e74c3c',
    'RandomForest':     '#e67e22',
    'Hybrid':           '#16a085',
}

_MONTH_STARTS = [0, 31, 61, 92, 122, 153, 181, 212, 243, 273, 304, 334]
_MONTH_LABELS = ['Oct','Nov','Dec','Jan','Feb','Mar',
                  'Apr','May','Jun','Jul','Aug','Sep']

def _month_ticks(n):
    t = [d for d in _MONTH_STARTS if d < n]
    l = [_MONTH_LABELS[i] for i, d in enumerate(_MONTH_STARTS) if d < n]
    return t, l

def _style_ax(ax, n, ylabel='', xlabel=False):
    ax.set_xlim(0, n - 1)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    t, l = _month_ticks(n)
    ax.set_xticks(t)
    ax.set_xticklabels(l if xlabel else [], fontsize=8)
    if xlabel:
        ax.set_xlabel('Month (season from Oct 1)', fontsize=9)

## 1. Load data

In [ ]:
cal      = Calendar(default_start='10-01', default_length=365)
features = OpenMeteoFeatures(calendar=cal)

ds_ys = Dataset.load('GMU_Cherry_Japan_YS', calendar=cal, feature_providers=[features])
ds_ys.download_features(verbose=True)

ds_y = ds_ys.select_species([('GMU_cherry', 0)])   # Somei-yoshino

target_fn = lambda s: s['observations'][OBS_KEY]

years_trn = [y for y in ds_y.years if y < CUTOFF]
years_tst = [y for y in ds_y.years if y >= CUTOFF]

ds_trn = ds_y.select_years(years_trn)
ds_tst = ds_y.select_years(years_tst)

print(f'Train: {len(ds_trn)} samples  ({min(years_trn)}–{max(years_trn)})')
print(f'Test:  {len(ds_tst)} samples  ({min(years_tst)}–{max(years_tst)})')

## 2. Train LSTM

Single meteorological feature (`temperature_2m_mean`). Training uses binary
cross-entropy against a soft step-function label so the model learns a smooth
cumulative-probability curve rather than a single-day spike.

In [ ]:
LSTM_KWARGS = dict(
    data_keys      = ['temperature_2m_mean'],
    hidden_size    = 64,
    num_layers     = 2,
    num_layers_lin = 2,
)

lstm, fit_info = LSTMModel.fit(
    target_fn        = target_fn,
    dataset          = ds_trn,
    model_kwargs     = LSTM_KWARGS,
    num_epochs       = 1000,
    batch_size       = 512,
    val_period       = 10,
    optimizer        = 'adam',
    optimizer_kwargs = dict(lr=1e-3, weight_decay=1e-4),
    scheduler_step_size      = 100,
    scheduler_decay          = 0.9,
    early_stopping           = True,
    early_stopping_patience  = 10,
    early_stopping_min_delta = 1e-3,
    early_stopping_rerun     = True,
    device  = DEVICE,
    verbose = True,
)
print('Training complete.')

In [ ]:
# Training / validation loss curves
epochs    = [e['epoch'] for e in fit_info['epochs']]
trn_loss  = [e['loss']  for e in fit_info['epochs']]
val_ep    = [e['epoch'] for e in fit_info['epochs'] if 'val' in e]
val_loss  = [e['val']['loss'] for e in fit_info['epochs'] if 'val' in e]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(epochs, trn_loss, label='Train', color='steelblue', lw=1.5)
if val_loss:
    ax.plot(val_ep, val_loss, label='Val', color='coral', lw=1.5)
    ax.axhline(min(val_loss), color='grey', ls='--', lw=0.8)
ax.set_xlabel('Epoch', fontsize=9)
ax.set_ylabel('BCE Loss', fontsize=9)
ax.set_title('LSTM training loss', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Evaluate

In [ ]:
result = SingleTargetRegression.run(
    model        = lstm,
    dataset_train= ds_trn,
    dataset_test = ds_tst,
    target_fn    = target_fn,
    run_name     = 'lstm_yedoensis',
)

m = result.compute_metrics()
rows = []
for split in ('train', 'test'):
    rows.append({
        'Split': split,
        'N':     m[split]['n'],
        'RMSE':  round(m[split]['rmse'], 2),
        'MAE':   round(m[split]['mae'],  2),
        'Bias':  round(m[split]['bias'], 2),
        'R²':    round(m[split]['r2'],   3),
    })
pd.DataFrame(rows).set_index('Split')

In [ ]:
fig = result.plot_scatter(title='LSTM — Somei-yoshino — predicted vs observed')
plt.tight_layout()
plt.show()

fig = result.plot_residuals_over_time(title='LSTM — Somei-yoshino — residuals over time')
plt.tight_layout()
plt.show()

## 4. LSTM probability curves on real samples

The LSTM outputs a per-day sigmoid probability `ps[t]` — the probability that
the bloom event has occurred by day *t*.  The predicted day is where this
curve rises fastest (argmax of the first difference).

In [ ]:
def get_lstm_response(model: LSTMModel, sample: dict) -> dict:
    """Run the LSTM on one sample and return the probability curve and prediction."""
    model.eval()
    with torch.no_grad():
        tensor_sample = LSTMModel.cast_to_tensor(sample)
        batch = LSTMModel.collate_fn([tensor_sample])
        ixs, info = model(batch)
    ps  = info['ps'][0].numpy()          # (T,)
    ix  = int(ixs[0].item() + 0.5)
    diff = np.diff(ps, prepend=ps[0])    # first difference
    return dict(ps=ps, diff=diff, ix=ix)


def pick_sample(ds, year):
    for item in ds.iter_items():
        if item['year'] == year:
            return item
    raise ValueError(f'No sample for year {year}')


# Pick three test years spanning early / middle / late bloom
test_obs = [
    (year,
     int((np.datetime64(pick_sample(ds_tst, year)['observations'][OBS_KEY], 'D')
          - np.datetime64(f'{year}-10-01', 'D')) / np.timedelta64(1, 'D')))
    for year in sorted(ds_tst.years)
]
test_obs.sort(key=lambda x: x[1])
n_tst = len(test_obs)
years_to_show = [
    test_obs[0][0],
    test_obs[n_tst // 2][0],
    test_obs[-1][0],
]
labels_to_show = ['Earliest bloom', 'Median bloom', 'Latest bloom']
print('Showing years:', years_to_show)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 9), sharex=False)
fig.suptitle('LSTM probability curves — test samples (Somei-yoshino)',
             fontsize=12, fontweight='bold')

for col, (year, lbl) in enumerate(zip(years_to_show, labels_to_show)):
    sample = pick_sample(ds_tst, year)
    ts     = sample[KEY_FEATURES]['temperature_2m_mean'].astype(float)
    resp   = get_lstm_response(lstm, sample)
    n      = len(ts)
    days   = np.arange(n)

    # Observed event day
    obs_date = sample['observations'][OBS_KEY]
    obs_ix = int((np.datetime64(obs_date, 'D')
                  - np.datetime64(sample['season_start'], 'D'))
                 / np.timedelta64(1, 'D'))

    # Row 0: temperature
    ax = axes[0, col]
    ax.axhline(0, color=PALETTE['zero'], lw=0.8)
    ax.fill_between(days, ts, 0, where=(ts >= 0), color=PALETTE['temp_warm'])
    ax.fill_between(days, ts, 0, where=(ts < 0),  color=PALETTE['temp_cold'])
    ax.plot(days, ts, color=PALETTE['temp_line'], lw=0.9)
    ax.axvline(obs_ix,    color='black',                  lw=1.2, ls='-',  alpha=0.7)
    ax.axvline(resp['ix'],color=MODEL_COLORS['LSTM'],     lw=1.2, ls='--', alpha=0.9)
    ax.set_title(f'{lbl}  (year {year})', fontsize=9, fontweight='bold')
    _style_ax(ax, n, ylabel='T (°C)' if col == 0 else '')

    # Row 1: sigmoid probability curve
    ax = axes[1, col]
    ax.plot(days, resp['ps'], color=MODEL_COLORS['LSTM'], lw=1.8)
    ax.fill_between(days, resp['ps'], color=MODEL_COLORS['LSTM'], alpha=0.12)
    ax.axhline(0.5, color=PALETTE['thresh'], lw=0.8, ls=':')
    ax.axvline(obs_ix,    color='black',              lw=1.2, ls='-',  alpha=0.7,
               label=f'Observed (day {obs_ix})')
    ax.axvline(resp['ix'],color=MODEL_COLORS['LSTM'], lw=1.2, ls='--', alpha=0.9,
               label=f'Predicted (day {resp["ix"]})')
    _style_ax(ax, n, ylabel='P(bloom by day t)' if col == 0 else '')
    ax.set_ylim(-0.05, 1.1)
    if col == 2:
        ax.legend(fontsize=7.5, loc='upper left', framealpha=0.85)

    # Row 2: first difference (where probability rises fastest)
    ax = axes[2, col]
    ax.bar(days, resp['diff'], width=1.0,
           color=MODEL_COLORS['LSTM'], alpha=0.7)
    ax.axvline(obs_ix,    color='black',              lw=1.2, ls='-',  alpha=0.7)
    ax.axvline(resp['ix'],color=MODEL_COLORS['LSTM'], lw=1.2, ls='--', alpha=0.9)
    _style_ax(ax, n, ylabel='dP/dt' if col == 0 else '', xlabel=True)

plt.tight_layout()
plt.show()

## 5. Sensitivity analysis on synthetic data

We generate synthetic seasonal temperature series across a sweep of mean winter
temperatures (0 → 16 °C), pass them through both the process-based models and
the trained LSTM, and compare the predicted bloom day — replicating the
sensitivity plot from `model_exploration.ipynb` with the LSTM added.

In [ ]:
def generate_temperature_series(
    n_days: int = 365,
    mean_winter_temp: float = 4.0,
    mean_summer_temp: float = 20.0,
    noise_std: float = 3.0,
    seed: Optional[int] = None,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mid_winter_day = 92
    days = np.arange(n_days)
    phase = 2 * np.pi * (days - mid_winter_day) / 365.0
    amplitude = (mean_summer_temp - mean_winter_temp) / 2.0
    baseline  = (mean_summer_temp + mean_winter_temp) / 2.0
    seasonal  = baseline - amplitude * np.cos(phase)
    return (seasonal + rng.normal(0, noise_std, size=n_days)).astype(np.float32)


def compute_pb_bloom_day(
    temps: np.ndarray,
    chill_fn,
    t_base: float,
    threshold_c: float,
    threshold_f: float,
) -> int:
    """Predict bloom day index for a process-based model on one temperature series."""
    n = len(temps)
    if chill_fn is not None:
        chill_cum = np.cumsum(chill_fn(temps))
        idx = np.where(chill_cum >= threshold_c)[0]
        fulfill = int(idx[0]) if len(idx) else n
    else:
        fulfill = 0
    force = np.maximum(temps - t_base, 0.0)
    force[:fulfill] = 0.0
    force_cum = np.cumsum(force)
    idx2 = np.where(force_cum >= threshold_f)[0]
    return int(idx2[0]) if len(idx2) else n


def make_synthetic_sample(temps: np.ndarray, year: int = 2000) -> dict:
    """Wrap a synthetic temperature array in a minimal sample dict."""
    return {
        KEY_FEATURES: {'temperature_2m_mean': temps},
        'season_start': np.datetime64(f'{year}-10-01', 'D'),
        'year': year,
    }


# Process-based models (fixed / literature parameters)
PB_MODELS = {
    'GDD': dict(chill_fn=None,             t_base=5.0, threshold_c=0.0,  threshold_f=200.0),
    'Utah+GDD':         dict(chill_fn=func_utah_chill,                          t_base=5.0, threshold_c=50.0,  threshold_f=200.0),
    'ChillingDays+GDD': dict(chill_fn=lambda t: func_chilling_days(t, 7.2),     t_base=5.0, threshold_c=50.0,  threshold_f=200.0),
    'Dynamic+GDD':      dict(chill_fn=lambda t: func_dynamic_chill_daily(t, 5), t_base=5.0, threshold_c=30.0,  threshold_f=200.0),
}

WINTER_TEMPS = np.arange(0.0, 17.0, 0.5)
N_SEEDS = 10

# --- Run sweep ---
pb_bloom   = {name: [] for name in PB_MODELS}
lstm_bloom = []

lstm.eval()
for mw in WINTER_TEMPS:
    pb_seeds   = {name: [] for name in PB_MODELS}
    lstm_seeds = []

    for seed in range(N_SEEDS):
        ts = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=seed)

        for name, m in PB_MODELS.items():
            pb_seeds[name].append(
                compute_pb_bloom_day(ts, m['chill_fn'], m['t_base'],
                                     m['threshold_c'], m['threshold_f'])
            )

        sample = make_synthetic_sample(ts)
        resp   = get_lstm_response(lstm, sample)
        lstm_seeds.append(resp['ix'])

    for name in PB_MODELS:
        pb_bloom[name].append(np.median(pb_seeds[name]))
    lstm_bloom.append(np.median(lstm_seeds))

lstm_bloom = np.array(lstm_bloom)
print('Sweep done.')

In [ ]:
# Month reference lines (approx. DOY from Oct 1)
_MONTH_REFS = [(92, 'Jan'), (153, 'Mar'), (181, 'Apr'), (212, 'May')]

fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle('Predicted bloom day vs. mean winter temperature\n'
             '(median over 10 synthetic seasons)',
             fontsize=12, fontweight='bold')

for doy, month in _MONTH_REFS:
    ax.axhline(doy, color='lightgrey', lw=0.7, zorder=0)
    ax.text(16.4, doy, month, fontsize=7.5, va='center', color='grey')

# Process-based models
for name, m in PB_MODELS.items():
    bloom = np.array(pb_bloom[name])
    valid = bloom < 364
    ax.plot(WINTER_TEMPS[valid], bloom[valid],
            color=MODEL_COLORS[name], lw=2.0, label=name)

# LSTM
valid_lstm = lstm_bloom < 364
ax.plot(WINTER_TEMPS[valid_lstm], lstm_bloom[valid_lstm],
        color=MODEL_COLORS['LSTM'], lw=2.5, ls='-',
        label='LSTM  (trained on yedoensis)')

ax.set_xlabel('Mean winter temperature (°C)', fontsize=10)
ax.set_ylabel('Predicted bloom day  (day of season from Oct 1)', fontsize=10)
ax.set_title('Predicted bloom day', fontsize=11)
ax.legend(fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 16.5)
plt.tight_layout()
plt.show()

## 6. LSTM probability curves on synthetic data

Show how the probability curve `ps[t]` — the LSTM equivalent of cumulative
forcing — changes shape across the winter temperature sweep.

In [ ]:
SCENARIO_TEMPS = [0.0, 4.0, 8.0, 12.0]
cmap_blues = plt.cm.Blues

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
fig.suptitle('LSTM probability curves on synthetic temperature series',
             fontsize=12, fontweight='bold')

for ax, mw in zip(axes.ravel(), SCENARIO_TEMPS):
    ts   = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=42)
    resp = get_lstm_response(lstm, make_synthetic_sample(ts))
    n    = len(ts)
    days = np.arange(n)

    ax2 = ax.twinx()

    # Temperature (right axis)
    ax2.plot(days, ts, color=PALETTE['temp_line'], lw=0.8, alpha=0.4)
    ax2.set_ylabel('T (°C)', fontsize=8, color='grey')
    ax2.tick_params(labelsize=7, colors='grey')
    ax2.spines['top'].set_visible(False)

    # LSTM probability curve (left axis)
    ax.plot(days, resp['ps'], color=MODEL_COLORS['LSTM'], lw=2.0, zorder=3,
            label='P(bloom by day t)')
    ax.fill_between(days, resp['ps'], color=MODEL_COLORS['LSTM'], alpha=0.12)
    ax.axhline(0.5, color=PALETTE['thresh'], lw=0.8, ls=':')
    ax.axvline(resp['ix'], color=MODEL_COLORS['LSTM'], lw=1.5, ls='--',
               label=f'Predicted bloom day {resp["ix"]}')

    _style_ax(ax, n, ylabel='P(bloom by day t)', xlabel=True)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(f'Mean winter = {mw:.0f} °C', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left', framealpha=0.85)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

plt.tight_layout()
plt.show()

## 7. LSTM response overlay across all winter temperatures

Show the probability curves for all synthetic scenarios on a single axis,
coloured by winter temperature — the LSTM equivalent of a response envelope.

In [ ]:
SWEEP_TEMPS = np.arange(0.0, 17.0, 2.0)
cmap = plt.cm.coolwarm_r
norm = plt.Normalize(SWEEP_TEMPS.min(), SWEEP_TEMPS.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM probability curves across winter temperatures  (seed 42)',
             fontsize=12, fontweight='bold')

ax_ps   = axes[0]
ax_diff = axes[1]

for mw in SWEEP_TEMPS:
    ts   = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=42)
    resp = get_lstm_response(lstm, make_synthetic_sample(ts))
    n    = len(ts)
    days = np.arange(n)
    color = cmap(norm(mw))

    ax_ps.plot(days, resp['ps'], color=color, lw=1.5, alpha=0.85,
               label=f'{mw:.0f} °C  (day {resp["ix"]})')
    ax_diff.plot(days, resp['diff'], color=color, lw=1.5, alpha=0.85)

ax_ps.axhline(0.5, color=PALETTE['thresh'], lw=0.8, ls=':')
_style_ax(ax_ps, n, ylabel='P(bloom by day t)', xlabel=True)
ax_ps.set_ylim(-0.05, 1.1)
ax_ps.set_title('Cumulative probability (sigmoid output)', fontsize=10)
ax_ps.legend(fontsize=7.5, ncol=2, framealpha=0.85, title='Winter temp')

ax_diff.axhline(0, color=PALETTE['zero'], lw=0.8)
_style_ax(ax_diff, n, ylabel='dP/dt  (event probability density)', xlabel=True)
ax_diff.set_title('First difference (where bloom probability rises fastest)', fontsize=10)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax_diff, label='Mean winter temp (°C)', shrink=0.85)

plt.tight_layout()
plt.show()

## 8. Random Forest model

Train a Random Forest on the same training set.  Features are the full season
of daily `temperature_2m_mean` and `daylight_duration` (in hours) flattened
into a single vector per sample — a simple bag-of-days representation that
gives the RF access to the full seasonal context without any temporal
inductive bias.

In [ ]:
RF_KWARGS = dict(
    data_keys         = ['temperature_2m_mean'],
    n_estimators      = 300,
    max_depth         = 15,
    min_samples_split = 4,
    random_state      = 42,
)

rf, rf_fit_info = RandomForestModel.fit(
    target_fn    = target_fn,
    dataset      = ds_trn,
    model_kwargs = RF_KWARGS,
)
print(f'RF fitted on {rf_fit_info["n_samples"]} samples, '
      f'feature shape {rf_fit_info["feature_shape"]}')

In [ ]:
result_rf = SingleTargetRegression.run(
    model         = rf,
    dataset_train = ds_trn,
    dataset_test  = ds_tst,
    target_fn     = target_fn,
    run_name      = 'rf_yedoensis',
)

m_rf = result_rf.compute_metrics()
rows_rf = []
for split in ('train', 'test'):
    rows_rf.append({
        'Split': split,
        'N':     m_rf[split]['n'],
        'RMSE':  round(m_rf[split]['rmse'], 2),
        'MAE':   round(m_rf[split]['mae'],  2),
        'Bias':  round(m_rf[split]['bias'], 2),
        'R²':    round(m_rf[split]['r2'],   3),
    })
pd.DataFrame(rows_rf).set_index('Split')

In [ ]:
fig = result_rf.plot_scatter(title='Random Forest — Somei-yoshino — predicted vs observed')
plt.tight_layout()
plt.show()

fig = result_rf.plot_residuals_over_time(title='Random Forest — Somei-yoshino — residuals over time')
plt.tight_layout()
plt.show()

### LSTM vs Random Forest metrics comparison

In [ ]:
m_lstm = result.compute_metrics()

rows_cmp = []
for model_name, m in [('LSTM', m_lstm), ('RandomForest', m_rf)]:
    for split in ('train', 'test'):
        rows_cmp.append({
            'Model': model_name,
            'Split': split,
            'N':     m[split]['n'],
            'RMSE':  round(m[split]['rmse'], 2),
            'MAE':   round(m[split]['mae'],  2),
            'Bias':  round(m[split]['bias'], 2),
            'R²':    round(m[split]['r2'],   3),
        })

pd.DataFrame(rows_cmp).set_index(['Model', 'Split'])

## 9. Sensitivity analysis — adding Random Forest

Extend the synthetic winter-temperature sweep from section 5 with the Random
Forest predictions.  The RF here uses only `temperature_2m_mean` (same as the
LSTM) so that both data-driven models respond to the synthetic temperature
variation rather than being anchored by a daylight signal that doesn't change
across the sweep.

In [ ]:
rf_bloom = []
for mw in WINTER_TEMPS:
    rf_seeds = []
    for seed in range(N_SEEDS):
        ts = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=seed)
        sample = make_synthetic_sample(ts)
        _, info = rf.predict(sample)
        rf_seeds.append(info['ix'])
    rf_bloom.append(np.median(rf_seeds))

rf_bloom = np.array(rf_bloom)
print('RF sweep done.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle('Predicted bloom day vs. mean winter temperature\n'
             '(median over 10 synthetic seasons)',
             fontsize=12, fontweight='bold')

for doy, month in _MONTH_REFS:
    ax.axhline(doy, color='lightgrey', lw=0.7, zorder=0)
    ax.text(16.4, doy, month, fontsize=7.5, va='center', color='grey')

# Process-based models
for name, m in PB_MODELS.items():
    bloom = np.array(pb_bloom[name])
    valid = bloom < 364
    ax.plot(WINTER_TEMPS[valid], bloom[valid],
            color=MODEL_COLORS[name], lw=2.0, label=name)

# LSTM
valid_lstm = lstm_bloom < 364
ax.plot(WINTER_TEMPS[valid_lstm], lstm_bloom[valid_lstm],
        color=MODEL_COLORS['LSTM'], lw=2.5, ls='-',
        label='LSTM  (trained on yedoensis)')

# Random Forest
valid_rf = rf_bloom < 364
ax.plot(WINTER_TEMPS[valid_rf], rf_bloom[valid_rf],
        color=MODEL_COLORS['RandomForest'], lw=2.5, ls='--',
        label='Random Forest  (trained on yedoensis)')

ax.set_xlabel('Mean winter temperature (°C)', fontsize=10)
ax.set_ylabel('Predicted bloom day  (day of season from Oct 1)', fontsize=10)
ax.legend(fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 16.5)
plt.tight_layout()
plt.show()

## 10. Unimodal Hybrid model

The Unimodal Hybrid model combines a **learned unimodal chilling stage** with a learned GDD forcing stage:

1. Two monotone-increasing networks `f_rise` and `f_fall` process normalised temperature pointwise → chilling contribution
   `tt_contrib = f_rise(T) × (1 − f_fall(T))`, which is **unimodal by construction**
2. Cumulative contributions are soft-thresholded → chilling fulfillment mask `tt_reached`
3. Daily GDD (`relu(T − t_base)`) is masked by `tt_reached`
4. Cumulative masked GDD is soft-thresholded → bloom probability curve `ps(t)`
5. Predicted day = argmax of first difference of `ps`

The key inductive bias: unlike a free-form CNN, the architecture can **only** represent bell-shaped (unimodal) chilling responses — matching the physical expectation that chilling accumulates most efficiently near an optimal temperature.

In [ ]:
UNIMODAL_KWARGS = dict(
    hidden_size      = 16,
    num_layers       = 3,
    learn_t_base     = False,
    t_base_init      = 5.0,
    learn_thresholds = True,
)

unimodal, unimodal_fit_info = UnimodalHybridModel.fit(
    target_fn        = target_fn,
    dataset          = ds_trn,
    model_kwargs     = UNIMODAL_KWARGS,
    num_epochs       = 1000,
    batch_size       = 512,
    val_period       = 10,
    optimizer        = 'adam',
    optimizer_kwargs = dict(lr=1e-3, weight_decay=1e-4),
    scheduler_step_size      = 100,
    scheduler_decay          = 0.9,
    early_stopping           = True,
    early_stopping_patience  = 10,
    early_stopping_min_delta = 1e-3,
    early_stopping_rerun     = True,
    device  = DEVICE,
    verbose = True,
)
print('Unimodal Hybrid training complete.')

In [ ]:
# Training / validation loss curves
epochs   = [e['epoch'] for e in unimodal_fit_info['epochs']]
trn_loss = [e['loss']  for e in unimodal_fit_info['epochs']]
val_ep   = [e['epoch'] for e in unimodal_fit_info['epochs'] if 'val' in e]
val_loss = [e['val']['loss'] for e in unimodal_fit_info['epochs'] if 'val' in e]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(epochs, trn_loss, label='Train', color='steelblue', lw=1.5)
if val_loss:
    ax.plot(val_ep, val_loss, label='Val', color='coral', lw=1.5)
    ax.axhline(min(val_loss), color='grey', ls='--', lw=0.8)
ax.set_xlabel('Epoch', fontsize=9)
ax.set_ylabel('BCE Loss', fontsize=9)
ax.set_title('Unimodal Hybrid model training loss', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Learned parameters
t_base  = unimodal._gdd._tb.item()
tt_thr  = unimodal._tt_unit_threshold.threshold.item()
gdd_thr = unimodal._gdd_threshold.threshold.item()
print(f't_base (fixed):             {t_base:.2f} °C')
print(f'Learned chilling threshold: {tt_thr:.3f} (normalised) '
      f'≈ {tt_thr * 200:.1f} cumulative chilling units')
print(f'Learned forcing threshold:  {gdd_thr:.3f} (normalised) '
      f'≈ {gdd_thr * 500:.1f} cumulative GDD')

In [ ]:
result_unimodal = SingleTargetRegression.run(
    model         = unimodal,
    dataset_train = ds_trn,
    dataset_test  = ds_tst,
    target_fn     = target_fn,
    run_name      = 'unimodal_yedoensis',
)

m_unimodal = result_unimodal.compute_metrics()
rows_all = []
for model_name, m in [('LSTM', result.compute_metrics()),
                       ('RandomForest', m_rf),
                       ('UnimodalHybrid', m_unimodal)]:
    for split in ('train', 'test'):
        rows_all.append({
            'Model': model_name, 'Split': split,
            'N':    m[split]['n'],
            'RMSE': round(m[split]['rmse'], 2),
            'MAE':  round(m[split]['mae'],  2),
            'Bias': round(m[split]['bias'], 2),
            'R²':   round(m[split]['r2'],   3),
        })
pd.DataFrame(rows_all).set_index(['Model', 'Split'])

## 11. Unimodal Hybrid model — temperature response investigation

A key advantage of the unimodal architecture: we can directly evaluate the learned chilling response on any temperature grid, without collecting statistics from training data.

We plot `f_rise(T)`, `1 − f_fall(T)`, and their product `tt_contrib(T)` across a temperature range to inspect the learned response curve, then show the full season dynamics for held-out years.

In [ ]:
@torch.no_grad()
def plot_chilling_response(model, t_range=(-5, 25), n=300):
    """Plot the learned unimodal chilling response vs temperature."""
    model.eval()
    device = next(model.parameters()).device
    t_mean, t_std = model._feature_statistics[model._temperature_key]

    t_grid = torch.linspace(t_range[0], t_range[1], n, device=device)
    t_norm = (t_grid - t_mean) / t_std
    dist   = (t_norm - model._T_opt_norm).abs().unsqueeze(0).unsqueeze(0)  # (1, 1, n)

    decay      = model._decay(dist)[0, 0].cpu().numpy()
    tt_contrib = 1.0 - decay
    t_vals     = t_grid.cpu().numpy()

    # Recover T_opt in raw °C
    T_opt_raw = model._T_opt_norm.item() * t_std + t_mean

    fig, ax = plt.subplots(figsize=(8, 4))
    fig.suptitle('Learned unimodal chilling response', fontsize=11, fontweight='bold')

    ax.plot(t_vals, decay,      color='steelblue',            lw=1.5, ls='--', label='decay(|T − T_opt|) — monotone ↑ in distance')
    ax.plot(t_vals, tt_contrib, color=MODEL_COLORS['Hybrid'], lw=2.5,          label='tt_contrib = 1 − decay')
    ax.axvline(T_opt_raw, color='grey', lw=1.0, ls=':', label=f'T_opt = {T_opt_raw:.1f} °C')
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_xlabel('Temperature (°C)', fontsize=10)
    ax.set_ylabel('Chilling contribution', fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    opt_ix = int(n // 2)  # by construction peak is at T_opt
    print(f'Learned optimal chilling temperature: {T_opt_raw:.1f} °C')
    print(f'Peak contribution:                    {tt_contrib[opt_ix]:.3f}')


plot_chilling_response(unimodal)

In [ ]:
@torch.no_grad()
def plot_unimodal_season(model, sample, obs_ix=None, title=''):
    """Five-panel plot of the unimodal hybrid model's internal dynamics."""
    model.eval()
    device = next(model.parameters()).device
    t_sample = UnimodalHybridModel.cast_to_tensor(sample, device=device)
    batch    = UnimodalHybridModel.collate_fn([t_sample])
    ixs, info = model(batch)

    pred_ix = int(ixs[0].item() + 0.5)
    ts      = sample[KEY_FEATURES]['temperature_2m_mean'].astype(float)
    n       = len(ts)
    days    = np.arange(n)

    tt_contrib = info['tt_contrib'][0].cpu().numpy()
    tt_units   = info['tt_units'][0].cpu().numpy()
    tt_reached = info['tt_reached'][0].cpu().numpy()
    gdd_cum    = info['gdd_cum'][0].cpu().numpy()
    ps         = info['ps'][0].cpu().numpy()

    tt_thr  = model._tt_unit_threshold.threshold.item()
    gdd_thr = model._gdd_threshold.threshold.item()
    t_base  = model._gdd._tb.item()

    color_c = MODEL_COLORS['Hybrid']
    color_f = '#c0392b'

    fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
    if title:
        fig.suptitle(title, fontsize=11, fontweight='bold')

    # 1. Temperature
    ax = axes[0]
    ax.axhline(0, color=PALETTE['zero'], lw=0.8)
    ax.axhline(t_base, color='navy', lw=1.0, ls=':', label=f't_base = {t_base:.1f} °C')
    ax.fill_between(days, ts, 0, where=(ts >= 0), color=PALETTE['temp_warm'])
    ax.fill_between(days, ts, 0, where=(ts < 0),  color=PALETTE['temp_cold'])
    ax.plot(days, ts, color=PALETTE['temp_line'], lw=0.9)
    if obs_ix is not None:
        ax.axvline(obs_ix,  color='black', lw=1.2, ls='-',  alpha=0.7, label='Observed')
    ax.axvline(pred_ix, color=color_c, lw=1.2, ls='--', alpha=0.9, label='Predicted')
    _style_ax(ax, n, ylabel='T (°C)')
    ax.legend(fontsize=7.5, loc='upper right')

    # 2. Unimodal chilling contribution
    ax = axes[1]
    ax.fill_between(days, tt_contrib, color=color_c, alpha=0.35)
    ax.plot(days, tt_contrib, color=color_c, lw=1.5)
    _style_ax(ax, n, ylabel='tt_contrib')
    ax.set_ylim(0, 1.05)
    ax.set_title('Learned chilling contribution (unimodal)', fontsize=8)

    # 3. Cumulative chilling + threshold
    ax = axes[2]
    ax.plot(days, tt_units, color=color_c, lw=1.8, label='Cumulative chilling')
    ax.axhline(tt_thr * 200, color=color_c, lw=1.0, ls=':',
               label=f'Threshold ≈ {tt_thr * 200:.1f}')
    ax2 = ax.twinx()
    ax2.plot(days, tt_reached, color=color_c, lw=1.0, ls='--', alpha=0.6)
    ax2.set_ylabel('tt_reached', fontsize=8, color=color_c)
    ax2.set_ylim(-0.05, 1.1)
    ax2.tick_params(labelsize=7, colors=color_c)
    _style_ax(ax, n, ylabel='Cum. chilling units')
    ax.legend(fontsize=7.5)

    # 4. Cumulative masked GDD
    ax = axes[3]
    ax.fill_between(days, gdd_cum, color=color_f, alpha=0.3)
    ax.plot(days, gdd_cum, color=color_f, lw=1.8, label='Cumulative masked GDD')
    ax.axhline(gdd_thr * 500, color=color_f, lw=1.0, ls=':',
               label=f'Threshold ≈ {gdd_thr * 500:.1f}')
    _style_ax(ax, n, ylabel='Cum. masked GDD')
    ax.legend(fontsize=7.5)

    # 5. Bloom probability
    ax = axes[4]
    ax.plot(days, ps, color=color_c, lw=2.0)
    ax.fill_between(days, ps, color=color_c, alpha=0.12)
    ax.axhline(0.5, color=PALETTE['thresh'], lw=0.8, ls=':')
    if obs_ix is not None:
        ax.axvline(obs_ix,  color='black', lw=1.2, ls='-',  alpha=0.7)
    ax.axvline(pred_ix, color=color_c, lw=1.2, ls='--', alpha=0.9)
    _style_ax(ax, n, ylabel='ps(t)', xlabel=True)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title('Bloom probability ps(t)', fontsize=8)

    plt.tight_layout()
    plt.show()


for year, lbl in zip(years_to_show, labels_to_show):
    sample = pick_sample(ds_tst, year)
    obs_date = sample['observations'][OBS_KEY]
    obs_ix   = int((np.datetime64(obs_date, 'D')
                    - np.datetime64(sample['season_start'], 'D'))
                   / np.timedelta64(1, 'D'))
    plot_unimodal_season(unimodal, sample, obs_ix=obs_ix,
                         title=f'Unimodal Hybrid — {lbl}  (year {year})')

## 12. All models — synthetic sensitivity sweep

Add the Unimodal Hybrid model to the winter-temperature sensitivity sweep from section 5. We expect it to show a qualitatively different response from the LSTM because its chilling stage is an explicit prerequisite before forcing can accumulate — and the unimodal temperature response ensures chilling only accumulates at physically plausible temperatures.

In [ ]:
unimodal_bloom = []
unimodal.eval()

for mw in WINTER_TEMPS:
    seeds = []
    for seed in range(N_SEEDS):
        ts     = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=seed)
        sample = make_synthetic_sample(ts)
        t_s    = UnimodalHybridModel.cast_to_tensor(sample)
        batch  = UnimodalHybridModel.collate_fn([t_s])
        ixs, _ = unimodal(batch)
        seeds.append(int(ixs[0].item() + 0.5))
    unimodal_bloom.append(np.median(seeds))

unimodal_bloom = np.array(unimodal_bloom)

# ---- Plot ----
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle('Predicted bloom day vs. mean winter temperature
'
             '(median over 10 synthetic seasons)',
             fontsize=12, fontweight='bold')

for doy, month in _MONTH_REFS:
    ax.axhline(doy, color='lightgrey', lw=0.7, zorder=0)
    ax.text(16.4, doy, month, fontsize=7.5, va='center', color='grey')

for name, m in PB_MODELS.items():
    bloom = np.array(pb_bloom[name])
    valid = bloom < 364
    ax.plot(WINTER_TEMPS[valid], bloom[valid],
            color=MODEL_COLORS[name], lw=2.0, label=name)

valid_lstm = lstm_bloom < 364
ax.plot(WINTER_TEMPS[valid_lstm], lstm_bloom[valid_lstm],
        color=MODEL_COLORS['LSTM'], lw=2.5, label='LSTM')

valid_rf = rf_bloom < 364
ax.plot(WINTER_TEMPS[valid_rf], rf_bloom[valid_rf],
        color=MODEL_COLORS['RandomForest'], lw=2.5, ls='--', label='Random Forest')

valid_u = unimodal_bloom < 364
ax.plot(WINTER_TEMPS[valid_u], unimodal_bloom[valid_u],
        color=MODEL_COLORS['Hybrid'], lw=2.5, label='Unimodal Hybrid')

ax.set_xlabel('Mean winter temperature (°C)', fontsize=10)
ax.set_ylabel('Predicted bloom day  (day of season from Oct 1)', fontsize=10)
ax.legend(fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 16.5)
plt.tight_layout()
plt.show()